# Reddit MCP

Connecting a third-party MCP server ([adhikasp/mcp-reddit](https://github.com/adhikasp/mcp-reddit)) to a LangChain agent.

Nothing is installed into this project. `uvx` fetches the server into its own cache and runs it as a
subprocess, so the server's 79 dependencies never touch our `pyproject.toml`. That isolation is the
whole point of MCP.

No Reddit credentials are needed - the server reads public listings anonymously.

In [ ]:
import asyncio
import sys

from dotenv import load_dotenv

load_dotenv()

# MCP stdio servers are subprocesses, which need the Proactor loop on Windows.
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

## Point the client at the server

The Smithery snippet on the directory page (`npx -y @smithery/cli install ...`) is an **installer** that
edits Claude Desktop's config and exits. It is not a server command - using it here would spawn a
process that never speaks MCP. The real command comes from the project README.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "reddit": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "--from",
                "git+https://github.com/adhikasp/mcp-reddit.git",
                "mcp-reddit",
            ],
        }
    }
)

## Discover what the server actually offers

Always list the tools before writing a prompt about them. The directory page advertises
`fetch_hot_threads`; the server really exposes `fetch_reddit_hot_threads`. A prompt naming the
documented name would describe a tool that does not exist.

The first run clones and builds the server, so give it a minute.

In [ ]:
tools = await client.get_tools()

for tool in tools:
    print(tool.name)
    print(f"  doc:  {tool.description.strip()}")
    print(f"  args: {tool.args}\n")

fetch_reddit_hot_threads
  doc:  Fetch hot threads from a subreddit
  args: {'subreddit': {'type': 'string', 'description': 'Name of the subreddit'}, 'limit': {'default': 10, 'type': 'integer', 'description': 'Number of posts to fetch (default: 10)'}}

fetch_reddit_post_content
  doc:  Fetch detailed content of a specific post
  args: {'post_id': {'type': 'string', 'description': 'Reddit post ID'}, 'comment_limit': {'default': 20, 'type': 'integer', 'description': 'Number of top level comments to fetch'}, 'comment_depth': {'default': 3, 'type': 'integer', 'description': 'Maximum depth of comment tree to traverse'}}



## Build the agent

Past this line there is nothing MCP-specific. `get_tools` returned ordinary LangChain tools, so the
agent is built exactly as in module 1. Same model and same key as the chef project.

The `limit` instruction in the system prompt is a cost control: every tool result re-enters the prompt
on every later turn, and Reddit listings are verbose.

In [4]:
from langchain.agents import create_agent

MODEL = "openrouter:gemini-3.5-flash-lite"

agent = create_agent(
    model=MODEL,
    tools=tools,
    system_prompt=(
        "You browse Reddit for the user using the tools provided. "
        "Keep limits small - never fetch more than 5 threads or 10 comments "
        "unless the user asks for more. Always cite thread titles."
    ),
)

In [5]:
response = await agent.ainvoke(
    {"messages": [("user", "What are the top 3 hot threads in r/LangChain right now?")]}
)

print(response["messages"][-1].content)

Here are the top 3 hot threads in r/LangChain right now:

1. **[Agent Plugins might be one of the more useful boring standards for AI agents.](https://reddit.com/r/LangChain/comments/1vtjmqb/agent_plugins_might_be_one_of_the_more_useful/)** (Score: 2)
2. **[Built a multi-agent LangGraph system for employee onboarding & offboarding with Azure OpenAI + human approval gate](https://reddit.com/r/LangChain/comments/1vthm0g/built_a_multiagent_langgraph_system_for_employee/)** (Score: 2)
3. **[What’s the point of LangGraph now that frontier AI providers are getting better at agent building?](https://reddit.com/r/LangChain/comments/1vssm2r/whats_the_point_of_langgraph_now_that_frontier_ai/)** (Score: 59)


## The message trace

Same four-message shape as module 1: Human, AI with empty content carrying `tool_calls`, ToolMessage
carrying the matching `tool_call_id`, then the final AI answer. The only difference is that the middle
step crossed a process boundary.

In [6]:
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

What are the top 3 hot threads in r/LangChain right now?
================================== Ai Message ==================================
Tool Calls:
  fetch_reddit_hot_threads (call_2904998)
 Call ID: call_2904998
  Args:
    subreddit: LangChain
    limit: 3
================================= Tool Message =================================
Name: fetch_reddit_hot_threads

[{'type': 'text', 'text': 'Title: Agent Plugins might be one of the more useful boring standards for AI agents.\nScore: 2\nComments: 0\nAuthor: ialijr\nType: unknown\nContent: None\nLink: https://reddit.comhttps://www.reddit.com/r/LangChain/comments/1vtjmqb/agent_plugins_might_be_one_of_the_more_useful/\n---\n\nTitle: Built a multi-agent LangGraph system for employee onboarding & offboarding with Azure OpenAI + human approval gate\nScore: 2\nComments: 2\nAuthor: Downey07\nType: text\nContent: Hey everyone,\n\n\n\nI built a multi-agent Lan

## Two tools, one chain

`fetch_reddit_hot_threads` returns post IDs; `fetch_reddit_post_content` takes one. Asking a question
that spans both makes the agent loop twice - list, pick, then drill in - without being told how.

In [7]:
response = await agent.ainvoke(
    {
        "messages": [
            (
                "user",
                "Find the most discussed thread in r/LangChain and summarise what "
                "people are actually arguing about in the comments.",
            )
        ]
    }
)

print(response["messages"][-1].content)

The most discussed thread right now in **r/LangChain** is titled: **"What’s the point of LangGraph now that frontier AI providers are getting better at agent building?"**

In the comments, users are actively debating whether framework-based orchestration is still necessary given how powerful native tools from OpenAI, Anthropic, and others have become. Here is what they are arguing about:

1. **Deterministic Control vs. "Jazz Improvisation"**
   * *Pro-LangGraph camp:* One user famously analogized frontier agent tooling to an "improvising jazz band" and LangGraph to "sheet music with stage cues." If you need an agent workflow to behave reliably, hit deterministic branches, integrate human-in-the-loop approval gates, and run identically for thousands of users at 3:00 AM, you need rigid orchestration rather than trusting pure model autonomy.
   * *Anti-LangGraph camp:* Some commenters argue that heavy graph frameworks are unnecessary and over-engineered complexity pushed largely by hype a

In [8]:
# How many round trips did that take, and what did it cost?
from langchain.messages import AIMessage, ToolMessage

tool_calls = sum(len(m.tool_calls) for m in response["messages"] if isinstance(m, AIMessage))
tool_chars = sum(len(str(m.content)) for m in response["messages"] if isinstance(m, ToolMessage))

print(f"messages:   {len(response['messages'])}")
print(f"tool calls: {tool_calls}")
print(f"tool result characters pulled into context: {tool_chars:,}")

messages:   6
tool calls: 2
tool result characters pulled into context: 8,770


## What to remember

- **Directory pages lie.** Tool names, arguments and descriptions come from `get_tools()`, not docs.
  Run the server, list the tools, then write the prompt.
- **An install command is not a server command.** Smithery's `npx ... install --client claude` writes a
  config file and exits.
- **`transport: stdio` means a real subprocess.** It inherits neither your venv nor your `.env`. Anything
  the server needs goes in an `env` key in its config block.
- **A third-party MCP server is arbitrary code you did not write**, pinned to a moving git branch and
  free to change its tools between runs. Its tool descriptions land in your prompt verbatim. Read the
  source before trusting one with anything that matters, and pin a commit for real work.
- **Tool output is the cost.** Reddit listings are long, and every result stays in context for the rest
  of the conversation.